# 1 — Data quality and cleaning

The source is the UCI *Online Retail II* workbook: two sheets of invoice-line
data from a UK online giftware retailer, Dec 2009 – Dec 2011.

This notebook documents what is wrong with the raw data and what the cleaning
layer does about it. The logic itself lives in `sql/01_staging_transactions.sql`
so it runs in the warehouse, not here — these cells only read and display it.

In [1]:
import pandas as pd

from retail_pricing.db import read_sql

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## The raw table

Loaded verbatim from the workbook by `retail_pricing.ingest` — no filtering, so
the raw layer stays auditable against the source file.

In [2]:
read_sql('''
    SELECT count(*) AS rows,
           count(DISTINCT invoice)     AS invoices,
           count(DISTINCT stock_code)  AS products,
           count(DISTINCT customer_id) AS customers,
           min(invoice_date)::date     AS first_day,
           max(invoice_date)::date     AS last_day
    FROM raw.online_retail
''')

,rows,invoices,products,customers,first_day,last_day
0,1067371,53628,5304,5942,2009-12-01,2011-12-09


## What is wrong with it

Five problems, each of which would distort a pricing model if ignored.

In [3]:
read_sql('''
    SELECT
        count(*) FILTER (WHERE customer_id IS NULL)  AS missing_customer,
        count(*) FILTER (WHERE description IS NULL)  AS missing_description,
        count(*) FILTER (WHERE invoice LIKE 'C%')    AS credit_notes,
        count(*) FILTER (WHERE quantity <= 0)        AS non_positive_quantity,
        count(*) FILTER (WHERE price <= 0)           AS non_positive_price
    FROM raw.online_retail
''')

,missing_customer,missing_description,credit_notes,non_positive_quantity,non_positive_price
0,243007,4382,19494,22950,6207


### 1. Duplicate lines

3.2% of rows repeat the same invoice, product, quantity, timestamp, price and
customer. Left in, they inflate the demand quantities the elasticity model is
fitted on, so they are collapsed in staging.

In [4]:
read_sql('''
    SELECT count(*) AS duplicate_groups, sum(c - 1) AS surplus_rows
    FROM (
        SELECT count(*) AS c
        FROM raw.online_retail
        GROUP BY invoice, stock_code, quantity, invoice_date, price, customer_id
        HAVING count(*) > 1
    ) d
''')

,duplicate_groups,surplus_rows
0,32909,34337.0


### 2. Not every StockCode is a product

Some codes are fees and adjustments — postage, bank charges, Amazon fees, gift
vouchers. They carry a price but have no demand curve.

The exclusion list was built by *reading* the irregular codes rather than by
pattern-matching, which matters: the `DCGS*` codes look like junk but are real
catalogue items, and a `^[0-9]{5}` regex would have silently dropped them.

In [5]:
read_sql('''
    SELECT stock_code,
           count(*)                            AS lines,
           round(sum(quantity * price)::numeric, 0) AS gross,
           min(description)                    AS example_description
    FROM raw.online_retail
    WHERE stock_code !~ '^[0-9]{5}'
    GROUP BY stock_code
    ORDER BY lines DESC
    LIMIT 20
''')

,stock_code,lines,gross,example_description
0,POST,2122,112341.0,POSTAGE
1,DOT,1446,322647.0,DOTCOM POSTAGE
2,M,1421,-82796.0,Manual
3,C2,282,13386.0,CARRIAGE
4,D,177,-13485.0,Discount
5,S,104,-6066.0,SAMPLES
6,BANK CHARGES,102,-35563.0,Bank Charges
7,ADJUST,67,6835.0,Adjustment by john on 26/01/2010 16
8,AMAZONFEE,43,-260764.0,AMAZON FEE
9,DCGS0058,31,35.0,MISO PRETTY GUM


In [6]:
read_sql("SELECT * FROM staging.non_product_codes ORDER BY stock_code")

,stock_code,reason
0,ADJUST,stock adjustment
1,ADJUST2,stock adjustment
2,AMAZONFEE,marketplace fee
3,B,adjust bad debt
4,BANK CHARGES,bank charges
5,C2,carriage
6,CRUK,charity commission
7,D,discount
8,DOT,dotcom postage
9,M,manual adjustment


## The cleaning funnel

Every row removed is accounted for. 1,067,371 raw lines become 1,003,355 clean
sales lines — 94.0% retained.

(The middle rows overlap: a credit note can also carry a negative quantity, so
they do not sum to the difference.)

In [7]:
read_sql("SELECT * FROM staging.line_audit")

,stage,lines
0,raw lines,1067371
1,after dedupe,1033034
2,credit notes,19104
3,non-product codes,5800
4,non-positive quantity,3393
5,non-positive price,6019
6,clean sales lines,1003355


## A caveat worth stating early

22.8% of rows have no `customer_id`. Those lines are kept for the pricing
analysis, which works at the product-week grain and does not need a customer.
They are necessarily dropped from the RFM segmentation, which does — so the
customer-level results describe only the 77% of the business that is attributed.

In [8]:
read_sql('''
    SELECT
        count(*)                                      AS clean_lines,
        count(*) FILTER (WHERE customer_id IS NULL)   AS unattributed_lines,
        round(100.0 * count(*) FILTER (WHERE customer_id IS NULL) / count(*), 1)
                                                      AS pct_unattributed,
        round(sum(line_revenue) FILTER (WHERE customer_id IS NULL)::numeric, 0)
                                                      AS unattributed_revenue
    FROM staging.transactions
''')

,clean_lines,unattributed_lines,pct_unattributed,unattributed_revenue
0,1003355,226761,22.6,2575279.0
